## Chargement des données

In [ ]:
import pandas as pd

from Modules.text_processor import TextProcessor

# readind the dataset
file_path = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-02-25/article_dat.csv'
df = pd.read_csv(file_path)

# instantiating the text processor
text_processor = TextProcessor(download=True)

In [ ]:
import sys

print(sys.executable)

In [ ]:
# Affichage des colonnes de la base de données
df.columns

## Traitements initiaux

In [ ]:
# Selection des colonnes de type chaine de caractères d'intérêt
target_col = ['study_type', 'data_source', 'abstract', 'study_aim', 'title']
target_col

In [ ]:
# Selection de la table d'intérêt
df.loc[:, target_col].sample(4, random_state=4)

In [ ]:
# Afficher le type d'étude que compte la base de données
print(df["study_type"].unique().tolist())
print(len(df["study_type"].unique().tolist()))

df.groupby("study_type").size()

On observe la présence de modalités dupliquées ou quasi identiques, différant uniquement par des variations de casse ou de typographie (par exemple : ``Retrospective cohort`` vs ``Retrospective Cohort``, ou ``Cross-sectional`` vs ``Cross-Sectional``). Il est donc nécessaire de mettre en place un prétraitement visant à uniformiser ces modalités, afin d’éviter une fragmentation artificielle des catégories.

**Prétraitement des chaînes de caractères**
- Conversion de l’ensemble des textes en minuscules
- Suppression de la ponctuation (virgules, points, points d’exclamation, etc.)
- Normalisation des espaces (remplacement des espaces multiples par un espace simple)
- Suppression des mots très fréquents et peu informatifs (stop words : of, the, and, etc.)
- Suppression des espaces en début et en fin de texte

In [ ]:
# Application d'un premier traitement aux colonnes d'intérêt
for col in target_col:
    df["processed_"+col] = df[col].apply(
        lambda x: None if " ".join(text_processor.preprocess(x)) == "" else " ".join(text_processor.preprocess(x)) # Reconstitution des phrases apres avoir réalisé le prétraitement sur les chaines de caractères
        # On s'assure de bien concerver NA lorsque l'information n'est pas disponible
    )

df.loc[:, ["processed_"+col for col in target_col]].sample(4, random_state=4)

In [ ]:
# Afficher le type d'étude que compte la base de données
print(df["processed_study_type"].unique().tolist())
print(len(df["processed_study_type"].unique().tolist()))

df.groupby("processed_study_type").size()

In [ ]:
# Mettons la 1ere lettre de chaque modalité en majuscule
df["processed_study_type"] = df["processed_study_type"].apply(
    lambda x: x.upper() if isinstance(x, str) and x == "rct"
    else x.title() if isinstance(x, str)
    else x
    )
df.groupby("processed_study_type").size()

La distribution des types d’études montre une forte prédominance des ``Retrospective Cohort``, qui représentent la grande majorité des articles (212), loin devant les autres catégories comme les ``Cross-Sectional`` (41) ou les ``Registry`` (23). Cela indique que le corpus est principalement composé d’études observationnelles rétrospectives, tandis que les études expérimentales (``RCT``) et analytiques comme les ``Case-Control`` sont nettement moins représentées.

In [ ]:
# Vérifions si on a des lignes ou on a des ""
for col in target_col:
    nb = df.loc[df["processed_"+col] == ""].shape[0]
    if nb != 0:
        print(col)

# Aucune des colonnes d'intérêt ne possède des cellules avec "" pour indiquer une donnée manquante

In [ ]:
# Les colonnes possédant des données manquantes parmis les colonnes d'intérêt
res = []
for col in target_col:
    nb = sum(df[col].isna())
    nb_trait =sum(df["processed_"+col].isna())
    res.append((col, nb, nb_trait))

res
# Ainsi, on constate que aucune donnée manquante n'a été rajouté au jeu de données

In [ ]:
# Visualisation des lignes n'ayant de type d'étude
df.loc[df["processed_study_type"].isna(), ["processed_"+col for col in target_col]]

On observe que les trois lignes ne comportant pas de type d’étude sont également celles qui présentent le plus de valeurs manquantes sur les variables d’intérêt. Néanmoins, la colonne ``abstract`` étant renseignée pour ces observations, celles-ci restent pertinentes, dans la mesure où le clustering des études sera réalisé à partir de cette variable.

## Analyse des `Abstracts`

In [ ]:
abstracts = df['processed_abstract'].dropna()
len(abstracts)

### Disctribution de la longueur des abstrats prétraité

In [ ]:
import matplotlib.pyplot as plt

plt.hist([len(abstract) for abstract in abstracts.str.split()])
plt.xlabel("Nombre de mots")
plt.ylabel("Fréquence")
plt.title("Distribution des longueurs des résumés")
plt.grid(True)
plt.show()

Il faut donc qu'on enlève les abstracts de moins de 50 mots. car il n'y en a pas de moins de 50.

In [ ]:
# removing abstracts with less than 50 words because we have some abstracts with very few words like 0 and we want to focus on more substantial abstracts for our analysis
# df = df[df['processed_abstracts'].str.split().apply(len) > 50]

### Frequences des mots dans le corpus

> **Le corpus des abstracts prétraités**

In [ ]:
df['processed_abstract'].head()

In [ ]:
all_absracts_cleaned = df['processed_abstract'].dropna()
all_absracts_cleaned = " ".join(all_absracts_cleaned)
dict_of_words_occurences = text_processor.count_word_frequencies(text=all_absracts_cleaned, top_n=10000, preprocess=False)
len(dict_of_words_occurences)

print(f"Après prétraitement, le corpus des abstracts contient {len(dict_of_words_occurences)} mots distincts.")
# 4664

In [ ]:
dict_of_words_occurences

In [ ]:
from Modules.graphic_drawer import GraphicDrawer

graphic_drawer = GraphicDrawer()
graphic_drawer.draw_bar_plot(
    dict_of_words_occurences, 
    top_n=20, 
    title="Top 20 des mots les plus fréquents dans les abstracts (version prétraitée)", 
    ylabel="Mot", 
    xlabel="Fréquence")

La distribution des fréquences met en évidence une forte concentration autour de certains termes dominants tels que ``women``, ``black``, ``white``, ``patients`` et ``cancer``, suggérant un focus sur la santé des femmes et les disparités raciales, tandis que d'autres mots faiblement fréquents traduit une diversité thématique plus large.

> **Le corpus des abstracts bruts (issus de la base de données)**

In [ ]:
all_absracts_raw = " ".join(df['abstract'].apply(lambda x: str(x)))
dict_of_words_occurences_raw_version = text_processor.count_word_frequencies(all_absracts_raw, top_n=1000000000, preprocess=False)
graphic_drawer.draw_bar_plot(
    dict_of_words_occurences_raw_version, 
    top_n=20, 
    title="Top 20 des mots les plus fréquents dans les abstracts bruts", 
    ylabel="Mot", 
    xlabel="Fréquence")

In [ ]:
len(dict_of_words_occurences_raw_version)
print(f"Sans traitement préalable, le corpus des abstracts contient {len(dict_of_words_occurences_raw_version)} mots distincts.")

On observe qu’avant le traitement du texte, des mots dits inutiles tels que `and, of, the, to, with, were, for, a, the` etc. sont présents dans le corpus. Après le traitement, ces mots disparaissent (`13 417 vs 4 664` mots). Leur forte présence dans le corpus n’est toutefois pas informative, car ils n’apportent pas de contenu sémantique pertinent.

**Ci-dessous, un nuage de mots des deux corpus d’abstracts (avant et après traitement)**

> Avant traitement

In [ ]:
graphic_drawer.draw_wordcloud_from_freq(dict_of_words_occurences_raw_version, title="Word Cloud of Raw Abstracts")

> Après traitement

In [ ]:
graphic_drawer.draw_wordcloud_from_freq(dict_of_words_occurences, title="Word Cloud of Processed Abstracts")

Avec le `TF-IDF`, on obtient des vecteurs de ``4664`` dimensions par abstract, ce qui est énorme et problématique. En effet, pour clusteriser/classifier nos articles sur la base des abstracts, il faut les vectoriser et calculer des métriques de similarité (distance, similarité cosinus, etc.) entre eux. Or, dans un espace de grande dimension, les distances et les normes deviennent moins informatives.

Pour cela, nous utiliserons ``SciBERT``, qui permet de pallier ce problème et fait même davantage : il tient compte de l’aspect sémantique des abstracts, au lieu de se baser uniquement sur le nombre d’occurrences. Le ``TF-IDF`` encode certes une forme de sémantique (comme vu avec la fréquence des mots), ce qui permet déjà **de comprendre globalement de quoi parle le corpus**, mais cette approche reste moins précise.

Par conséquent, pour utiliser ``SciBERT``, il faut respecter au mieux la structure sémantique des textes. Ainsi, supprimer les ``stopwords``, le ``tokenizer``, ainsi que certains éléments issus de ``regex`` comme les ``chiffres`` ou la ``ponctuation`` peut fortement altérer le sens des abstracts. Les autres traitements restent valables. Nous allons donc redéfinir notre classe afin d’adapter le ``preprocessing`` à la vectorisation avec ``SciBERT``.

# Vectorisation des abstracts avec SciBERT

## Formalisme et cas d’usage du transformer

---

## 1. Entrée du modèle (tokenization)

Soit un texte :

"deep learning for image classification"

Après tokenization avec SciBERT :

inputs["input_ids"] =
[101, 2784, 4083, 2005, 3746, 5579, 102]

inputs["attention_mask"] =
[1, 1, 1, 1, 1, 1, 1]

### Interprétation :

- 101 = token [CLS] (début de phrase)
- 102 = token [SEP] (fin de phrase)
- les autres = tokens du texte

---

## 1.1 Qu’est-ce que [PAD] ?

Le token [PAD] signifie padding (remplissage).

Il est utilisé lorsque plusieurs textes sont traités ensemble (batch), car ils n’ont pas la même longueur.

### Exemple :

"deep learning"  
"climate change impacts marine ecosystems"

Devient :

["deep", "learning", [PAD], [PAD], [PAD]]  
["climate", "change", "impacts", "marine", "ecosystems"]

### Rôle de [PAD] :

- rendre toutes les séquences de même longueur
- permettre le traitement en batch
- ne pas ajouter d’information sémantique

---

## 1.2 Attention mask

Pour éviter que [PAD] influence le modèle, on utilise un attention mask :

attention_mask:
[1, 1, 0, 0, 0]

- 1 = vrai token (important)
- 0 = padding (ignoré)

---

## 2. Sortie du modèle BERT

On applique :

outputs = model(**inputs)

Le modèle retourne :

outputs.last_hidden_state  
shape = (batch_size, sequence_length, hidden_size)

### Exemple concret :

shape = (1, 7, 768)

### Interprétation :

Chaque token devient un vecteur de dimension 768 :

token 1 ([CLS])      → vector (768)  
token 2 ("deep")     → vector (768)  
token 3 ("learning") → vector (768)  
...  
token 7 ([SEP])      → vector (768)

---

## 3. Pourquoi on ne peut pas utiliser directement BERT

On veut UN vecteur par phrase :

$$
(1, 768)
$$

Mais BERT donne :

$$
(1, 7, 768)
$$

Donc il faut une opération de réduction : pooling.

---

## 4. Mean pooling (idée mathématique)

Soit :

$$
H = [h_1, h_2, ..., h_n]
$$

où chaque $h_i \in \mathbb{R}^{768}$

Mean pooling :

$$
h_{\text{sentence}} = \frac{1}{n} \sum_{i=1}^{n} h_i
$$

---

## 4.1 Mean pooling avec attention mask

Sans attention mask, les tokens [PAD] influencent la moyenne.

Avec mask :

$$
h = \frac{\sum_{i=1}^{n} m_i h_i}{\sum_{i=1}^{n} m_i}
$$

où :

- $m_i = 1$ si token valide  
- $m_i = 0$ si [PAD]

---

## 5. Code correspondant

```python
token_embeddings = outputs.last_hidden_state

mask = attention_mask.unsqueeze(-1)
mask = mask.expand(token_embeddings.size())

masked_embeddings = token_embeddings * mask

summed = masked_embeddings.sum(dim=1)
counts = mask.sum(dim=1)

sentence_embedding = summed / counts
```

In [ ]:
text_processor_with_scibert = TextProcessor(use_scibert=True, download=False)
abstracts_scibert = df['abstract'].apply(
    lambda x: text_processor_with_scibert.preprocess(x)
)

In [ ]:
df['abstracts_scibert'] = abstracts_scibert
# df['abstracts_scibert'] = df['abstracts_scibert'].apply(lambda x: " ".join(x))
all_absracts_scibert = " ".join(df['abstracts_scibert'].apply(lambda x: str(x)))
dict_of_words_occurences_scibert_version = text_processor_with_scibert.count_word_frequencies(all_absracts_scibert, top_n=1000000000, preprocess=False)
len(dict_of_words_occurences_scibert_version)

In [ ]:
dict_of_words_occurences_scibert_version

On passe de ``4 664`` à ``13 414`` mots uniques dans le corpus. Le nombre de mots a donc fortement augmenté, se rapprochant du nombre de mots du texte non traité, qui est de ``13 417``.

On peut ici observer la présence de chiffres, notamment `95%`, correspondant à des intervalles de confiance ou à des tests statistiques, ainsi que de `stopwords (to, or, etc.)`, ce qui permet de conserver le contexte des données. Les trois mots manquants peuvent s’expliquer par la suppression des adresses `e-mails`, des `URLs`, des `balises HTML` et des espaces supplémentaires. Nous pouvons maintenant passer à la vectorisation.

## Interprétation du nuage de mots

In [ ]:
graphic_drawer.draw_wordcloud_from_freq(dict_of_words_occurences_scibert_version, title="Word Cloud of SciBERT-Processed Abstracts")

Nous pouvons déjà identifier plusieurs termes clés. L’interprétation est réalisée de manière itérative, de haut en bas :

* `women` est le terme le plus fréquent (1135 occurrences après suppression des stopwords), associé à `pregnancy` → **GENRE**

* `race`, `racial`, `black`, `white` → **RACE**

* `non-hispanic` → **ORIGINE ETHNIQUE**

* `treatment`, `patients`, `risk`, `survival`, `care`, `cancer` → forte dominance des études liées à la santé

* `disparities`, `associated`, `less`, `between`, `after` → cadre d’études comparatives

* `95%`, `CI`, `likely`, `odds`, `interval`, `factors` → modélisation statistique / inférence

---

Un premier aperçu du jeu de données suggère que, même sans connaissance préalable du corpus, celui-ci est constitué d’articles scientifiques médicaux en anglais. Ces articles portent principalement sur des études cliniques impliquant les variables de race, d’ethnie et de genre, avec un fort accent sur les méthodes de modélisation statistique et d’inférence.


## Vectorisation du texte

In [ ]:
import matplotlib.pyplot as plt

plt.hist([len(abstract) for abstract in abstracts_scibert.apply(lambda x: str(x).split())])
plt.xlabel("Nombre de mots")
plt.ylabel("Fréquence")
plt.title("Distribution de la longueur des abstracts")
plt.grid(True)
plt.show()

Suppression des abstracts avec moins de 50 mots après le traitement avec ``SciBERT`` pour se concentrer sur les abstracts plus substantiels.

In [ ]:
df = df[df['abstracts_scibert'].str.split().apply(len) > 0]

La version de `SciBERT` utilisée (gratuite) nous donne accès à des séquences de `512` tokens pour chaque élément de notre corpus, ce qui correspond à environ `250 mots`. Or, la distribution montre que certains abstracts vont jusqu’à `600` mots, avec un mode autour de `300` mots. Il est donc nécessaire de procéder à un *chunking* afin d’utiliser plusieurs fenêtres de `512` tokens sur un même élément du corpus, c’est-à-dire un abstract, si son nombre de mots dépasse `250` (et nécessite donc plus de `512` tokens).


In [ ]:
texts = df['abstracts_scibert'].tolist()
print(f'Nous travaillons maintenant sur {len(texts)} abstracts après avoir supprimé ceux qui ne contiennent aucun mot.')

In [ ]:
embeddings = text_processor_with_scibert.embed_scibert(texts=texts, batch_size=16) # 16 because i do not know if the user will execute this code using a GPU

In [ ]:
# pre-commit install
embeddings.shape

In [ ]:
df.columns

In [ ]:
len(embeddings.tolist())

In [ ]:
df_4_umap = df[['study_type']].copy()

# convert the embeddings column into a DataFrame
emb_df = pd.DataFrame(embeddings.tolist(), index=df.index)

# concatenate
df_4_umap = pd.concat([df_4_umap, emb_df], axis=1)

# Optional: set `study_type` as the index
df_4_umap.set_index('study_type', inplace=True)

# Removing N/A entries from the study type
df_4_umap = df_4_umap[~df_4_umap.index.isna()]
print(len(df_4_umap))

In [ ]:
df_4_umap

## Optimisation Bayésienne

In [ ]:
import warnings

warnings.filterwarnings('ignore') # y'a juste un warning qui parle de redefinition du seed

In [ ]:
from rich.progress import (
    BarColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)

# for progress bar
from skopt.space import Integer, Real
from skopt.utils import use_named_args

from Modules.bayesian_optimizaion import BayesianOptimization
from Modules.hdbscan import HdbscanClusterer

# df for clustering
df_4_clustering = df_4_umap 

In [ ]:
SEED = 42
# hyperparameters space
min_cluster_size = [5, 30]
min_dist = [0.01, 0.5]
min_sample_ratio = [0.1, 0.5]
n_components = [5, 30]
n_neighbors = [5, 30]

# hyperparameters's ressearch space specifying
space = [
    Integer(min(n_neighbors), max(n_neighbors), name="n_neighbors"),
    Integer(min(n_components), max(n_components), name="n_components"),
    Integer(min(min_cluster_size), max(min_cluster_size), name="min_cluster_size"),
    Real(min(min_sample_ratio), max(min_sample_ratio), name="min_samples_ratio"),
    Real(min(min_dist), max(min_dist), name="min_dist"),
]


progress = Progress(
    SpinnerColumn(),
    "[progress.description]{task.description}",
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn()
)

iteration = 0
total_calls = 250

task = progress.add_task(
    "[green]Running Bayesian Optimization...",
    total=total_calls
)

@use_named_args(space)
def objective(n_neighbors, n_components, min_cluster_size, min_samples_ratio, min_dist):
    global iteration
    iteration += 1
    progress.update(task, advance=1)

    # Fit HDBSCAN + UMAP
    clusterer = HdbscanClusterer(metric="euclidean")
    clusterer.fit(
        data=df_4_clustering,
        n_neighbors=int(n_neighbors),
        n_components=int(n_components),
        min_cluster_size=int(min_cluster_size),
        min_dist=min_dist,
        min_samples_ratio=min_samples_ratio
    )

    # Récupération des métriques
    return -clusterer.best_scores["metric_val"]

# ------------------------------
# Lancement de l'optimisation
# ------------------------------
bopt_all_data = BayesianOptimization(
    objective_func=objective,
    space=space,
    n_calls=total_calls,
    n_initial_points=10,
    acq_func="EI",
    random_state=SEED,
)

with progress:
    bopt_all_data.run_optimization()

**Recherche des paramètres optimaux pour le clustering des articles**

In [ ]:
import numpy as np

best_params_bopt = {k: int(v) if isinstance(v, np.integer) else v for k, v in bopt_all_data.best_params.items()}
best_dbcv = bopt_all_data.best_score

best_params_clean = {
    k: int(v) if isinstance(v, (np.integer,))
    else float(v) if isinstance(v, (np.floating,))
    else v
    for k, v in best_params_bopt.items()
}
print(best_params_clean)

In [ ]:
# Résultat du chunk ci-dessus
# best_params_clean = {'n_neighbors': 23, 'n_components': 23, 'min_cluster_size': 13, 'min_samples_ratio': 0.43017338773314195, 'min_dist': 0.023209924063945047}

**Mise en œuvre du clustering à partir des paramètres obtenus par optimisation bayésienne**

In [ ]:
final_clusterer = HdbscanClusterer(metric="euclidean")
final_clusterer.fit(
    data=df_4_clustering,
    **best_params_clean
)
final_clusterer.summary()

In [ ]:
result_clustering = final_clusterer.get_results()
result_clustering

En vue d’associer les clusters aux articles, il est nécessaire de disposer d’une colonne identifiante. Ainsi, ci-dessous, nous vérifions que la ``pmid`` constitue une colonne identifiante possible.

In [ ]:
print(sum(df["pmid"].astype(str).isna()))
print(df["pmid"].astype(str).nunique())

print(sum(df.loc[~df['study_type'].isna(), "pmid"].astype(str).isna()))
print(len(df.loc[~df['study_type'].isna(), "pmid"].astype(str)))
print(df.loc[~df['study_type'].isna(), "pmid"].astype(str).nunique())

print(type(df.loc[~df['study_type'].isna(), "pmid"].astype(str)))


In [ ]:
# On a bien une correspondance donc entre les clusters et la table
print(sum(df['study_type'].dropna() == df_4_umap.index))

# Ajout de la colonne identifiante
result_clustering["pmid"] = df.loc[~df['study_type'].isna(), "pmid"].astype(str).reset_index(drop=True)
result_clustering

result_clustering["pmid"].nunique()
result_clustering.sample(10, random_state=4)

Association des clusters aux articles

In [ ]:
# Ajout de la colonne renseignant les cluster
df["pmid"] = df["pmid"].astype(str)
df_cluster = df.merge(right=result_clustering, on="pmid", how="left")

# Vérification
df_cluster.loc[
    df_cluster["pmid"].isin(result_clustering["pmid"].sample(10, random_state=4).tolist())
]

## Clustering avec UMAP + HDBSCAN

Nous avons optimisé un pipeline de clustering basé sur une réduction de dimension avec UMAP suivie d’un clustering avec HDBSCAN, en utilisant une optimisation bayésienne.

---

### Paramètres optimaux

- $n\_neighbors = 23$
- $min\_dist = 0.023$
- $n\_components = 23$
- $min\_cluster\_size = 13$
- $min\_samples = 5$
- Méthode de sélection : $eom$

---

### Résultats

- Score de validation (DBCV) :  
  $$
  \text{DBCV} \approx 0.50
  $$

- Ratio de bruit :  
  $$
  \text{noise\_ratio} = \frac{N_{\text{bruit}}}{N_{\text{total}}} \approx 0.17
  $$

- Stabilité :  
  $$
  \text{stability} \approx 0.15
  $$

- Score composite :  
  $$
  \text{score} = \alpha \cdot \text{DBCV} - \beta \cdot \text{noise\_ratio} + \gamma \cdot \text{stability}
  $$

---

### Interprétation

- Un score DBCV proche de $0.5$ indique une structure de clusters cohérente.
- Un ratio de bruit de $17\%$ est faible dans le cadre d’un clustering avec HDBSCAN.
- Une stabilité relativement faible suggère que les clusters peuvent être sensibles aux variations des données ou des paramètres.

---

### Nombre de clusters

$$
k = 4
$$

Ce résultat suggère une structuration des données en quatre groupes principaux.

---

### Conclusion

Le pipeline permet d’identifier des clusters cohérents avec un niveau de bruit maîtrisé. Une amélioration de la stabilité pourrait être envisagée en ajustant les hyperparamètres ou la réduction de dimension.

## Typologie des études et interprétation des clusters

| Type d’étude           | Intervention | Temps      | Description détaillée                                                                 | Interprétation possible dans les clusters |
|------------------------|-------------|------------|----------------------------------------------------------------------------------------|-------------------------------------------|
| **RCT**                | Oui         | Futur      | Étude expérimentale avec assignation aléatoire des participants à un traitement ou contrôle. Permet d’inférer une causalité forte. | Peut former un cluster distinct car structure méthodologique très spécifique et vocabulaire technique (randomization, placebo, trial). |
| Cohort (prospective)   | Non         | Futur      | Suivi d’un groupe dans le temps pour observer l’apparition d’un événement selon une exposition. | Peut être proche des RCT mais avec un vocabulaire observationnel → cluster voisin. |
| Cohort (retrospective) | Non         | Passé      | Analyse de données historiques pour étudier une relation exposition–résultat. | Peut se regrouper avec case-control (logique rétrospective). |
| Case-control           | Non         | Passé      | Compare des individus malades et non malades pour identifier des facteurs de risque passés. | Souvent clusterisé avec cohort rétrospective (logique causale inversée, analyse a posteriori). |
| Cross-sectional        | Non         | Instantané | Étude à un instant donné, sans suivi, décrivant une population ou une prévalence. | Peut former un cluster distinct (absence de temporalité, vocabulaire descriptif). |
| Registry               | Non         | Variable   | Base de données structurée collectant des informations longitudinales ou transversales. | Peut être diffus ou associé à plusieurs clusters selon l’usage (souvent moins spécifique). |

---

## Interprétation avec $k = 4$ clusters (HDBSCAN)

Le clustering obtenu avec HDBSCAN suggère une structuration sémantique des études en **4 grands groupes** :

$$
k = 4
$$

### Hypothèse d’organisation des clusters (à verifier)

1. **Cluster 1 : Études expérimentales**
   - Principalement RCT
   - Forte homogénéité méthodologique
   - Vocabulaire spécifique (randomization, intervention)

2. **Cluster 2 : Études observationnelles prospectives**
   - Cohort prospective
   - Notion de suivi temporel vers le futur

3. **Cluster 3 : Études rétrospectives**
   - Cohort rétrospective + Case-control
   - Analyse a posteriori, logique causale inversée

4. **Cluster 4 : Études descriptives / structurelles**
   - Cross-sectional + Registry
   - Absence de dynamique temporelle forte
   - Données souvent utilisées à des fins descriptives

---

## Lecture globale

Le clustering ne reproduit pas exactement les catégories théoriques, mais :

- Il regroupe les études selon leur **similarité sémantique réelle**
- Il met en évidence :
  $$
  \text{expérimental} \quad vs \quad \text{observationnel} \quad vs \quad \text{descriptif}
  $$

- Certaines catégories (comme *Registry*) peuvent être **transversales** et donc moins bien séparées

---

## Conclusion

Le résultat $k = 4$ est cohérent avec une structuration naturelle des types d’études :

- séparation des approches méthodologiques
- regroupement des études proches conceptuellement
- simplification des catégories initiales en grandes familles

Cela confirme que les embeddings capturent bien la **sémantique des designs d’étude**.

## Verification

- Plot `UMAP` avec les clusters identifiés comme couleur
- Analyse qualitative des `titres` et des `study_aim` des articles des clusters (car analyser tous les abstracts serait vraiment couteux en temps)
- Labeliser les clusters obtenus en  grands groupes et voir leur corcordance avec les `study_type`, `data_source`
- Voir les articles classés comme bruit

**QUESTIONS POUR TOUS** : Est-il judicieux de ne considerer que les abstracts ou don devrait plutot faire une concatenation du `titre`, du `study_aim`, du `study_type` et de l'`abstract` ? Rien ne change dans le code juste var1 + var2 + var3 = text_final

### Proportion d'articles par cluster

In [ ]:
total = df_cluster.groupby(["cluster"]).size().reset_index(name="Total")
print(total)
print(total["Total"].sum())

On observe que le cluster le plus important en nombre d’articles parmi les quatre est le cluster 0 (cluster 1), avec 136 articles, suivi du cluster 1 (cluster 2), qui en compte 80. Le cluster -1, considéré comme du bruit, regroupe 53 articles.

### La proportion de chaque type d'étude par cluster

In [ ]:
prop_study_type = df_cluster.groupby(["cluster", "processed_study_type"]).agg(
    nb_articles = ("processed_study_type", "count")
)\
    .sort_values(["cluster", "nb_articles"], ascending = [True, False])\
        .reset_index().merge(right = total, on = "cluster", how = "left")

prop_study_type["proportion"] = round(100*prop_study_type["nb_articles"]/prop_study_type["Total"], 0)

    Cluster 1

In [ ]:
prop_study_type.loc[prop_study_type["cluster"] == 0]

Celui-ci regroupe essentiellement des articles de type ``Retrospective Cohort`` et ``Registry``. En outre ce cluster semble regrouper des articles portant sur des **études observationnelles rétrospectives basées sur des données existantes.**.
En effet, le point commun entre ces articles est sans doute le fait qu'il y a pas d'action (observationnelle) et que les données sont anciennes. 

    Cluster 2

In [ ]:
prop_study_type.loc[prop_study_type["cluster"] == 1]

Ce cluster regroupe essentiellement des articles de type ``Retrospective Cohort`` et ``Cross Sectional``. En outre ce cluster semble regrouper des articles portant sur des **études observationnelles rétrospectives ou descriptives basées sur des données existantes.**. 
En effet, il s'agit de données de déja collecté et aussi c'est une analyse faite en un temps donné dans le passé

    Cluster 3

In [ ]:
prop_study_type.loc[prop_study_type["cluster"] == 2]

Ce cluster regroupe majoritairement des articles de type ``Retrospective Cohort`` et ``Cross-Sectional``, avec une présence secondaire d’études de cohorte prospectives. En outre, ce cluster semble regrouper des articles portant sur des **études observationnelles, principalement rétrospectives et descriptives, avec une faible composante prospective**.

    Cluster 4

In [ ]:
prop_study_type.loc[prop_study_type["cluster"] == 3]

Ce cluster regroupe majoritairement des articles de type ``Retrospective Cohort`` et ``Cross-Sectional``, avec une faible proportion d’études randomisées contrôlées (RCT). Il semble ainsi correspondre à des **études observationnelles principalement rétrospectives et transversales, avec une composante expérimentale marginale**.

### La distribution de la longeur des abstracts par cluster

In [ ]:
import matplotlib.pyplot as plt

for clt in range(4):
    plt.hist([len(abstract) for abstract in df_cluster.loc[df_cluster["cluster"] == clt, "abstracts_scibert"].apply(lambda x: str(x).split())])
    plt.xlabel("Nombre de mots")
    plt.ylabel("Fréquence")
    plt.title(f"Distribution de la longueur des abstracts dans le cluster {clt+1}")
    plt.grid(True)
    plt.show()

On observe que la longueur des abstracts varie fortement d’un cluster à l’autre. Les clusters 1 et 3 se caractérisent par une majorité d’abstracts relativement courts, avec une longueur généralement comprise entre 200 et 300 mots. Le cluster 2, quant à lui, présente une distribution plus étalée, avec une proportion notable d’abstracts dépassant les 400 mots. Enfin, le cluster 4 se distingue par des effectifs plus faibles, mais une répartition des longueurs globalement homogène.

### Frequences des mots dans le corpus de chaque cluster

In [ ]:
from Modules.graphic_drawer import GraphicDrawer

for clt in range(4):
    all_absracts_cleaned = df_cluster.loc[df_cluster["cluster"] == clt, "processed_abstract"].dropna()
    all_absracts_cleaned = " ".join(all_absracts_cleaned)
    dict_of_words_occurences = text_processor.count_word_frequencies(text=all_absracts_cleaned, top_n=10000, preprocess=False)
    print(len(dict_of_words_occurences))

    graphic_drawer = GraphicDrawer()
    graphic_drawer.draw_bar_plot(
        dict_of_words_occurences, top_n=20, 
        title=f"Top 20 des mots les plus fréquents dans le cluster {1+clt}", 
        ylabel="Mot", xlabel="Fréquence")

# 4664

L’analyse des mots les plus fréquents met en évidence une structuration thématique claire des clusters. Le cluster 1 est principalement centré sur l’oncologie et les disparités raciales, tandis que le cluster 2 regroupe des études en santé maternelle, également marquées par des inégalités selon l’origine ethnique. Enfin, le cluster 3 apparaît plus transversal, regroupant des travaux portant sur les soins de santé, la génétique et la santé reproductive. Le cluster 4 quant à lui regroupe principalement des articles portant sur l’utilisation des méthodes contraceptives, en particulier les LARC, avec une analyse des disparités selon les caractéristiques démographiques et socio-économiques.

### Analyse du bruit - cluster -1

In [ ]:
prop_study_type.loc[prop_study_type["cluster"] == -1]

Le cluster -1, identifié comme du bruit, regroupe des articles de nature hétérogène, sans structure thématique claire. Bien qu’une majorité d’études de cohorte rétrospectives y soit observée, la diversité des types d’études suggère que ces articles ne présentent pas de similarités suffisantes pour être intégrés dans les autres clusters.

In [ ]:
all_absracts_cleaned = df_cluster.loc[df_cluster["cluster"] == -1, "processed_abstract"].dropna()
all_absracts_cleaned = " ".join(all_absracts_cleaned)
dict_of_words_occurences = text_processor.count_word_frequencies(text=all_absracts_cleaned, top_n=10000, preprocess=False)
print(len(dict_of_words_occurences))

graphic_drawer = GraphicDrawer()
graphic_drawer.draw_bar_plot(dict_of_words_occurences, top_n=20, title=f"Top 20 des mots les plus fréquents dans le cluster -1", ylabel="Mot", xlabel="Fréquence")
